In [ ]:
pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.5 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.nn import NNConv
from copy import deepcopy
from itertools import combinations
import matplotlib.pyplot as plt
import copy
import math

# === CVRP Grid Environment (unchanged) === #
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch
import random
import numpy as np
from torch_geometric.data import Data

class CVRPGridEnv:
    def __init__(self, grid_size=10, vehicle_capacity=15, num_customers=10, seed=None, device='cuda' if torch.cuda.is_available() else 'cpu', demand_noise_std=1.0):
        self.grid_size = grid_size
        self.num_nodes = grid_size * grid_size
        self.vehicle_capacity = vehicle_capacity
        self.num_customers = min(num_customers, self.num_nodes - 1)
        self.seed = seed
        self.device = torch.device(device)
        self.demand_noise_std = demand_noise_std  # Standard deviation for demand uncertainty

        self.traffic_transition_beta = 0.07
        self.traffic_noise_std = 1.0

        self.step_count = 0
        self.traffic_update_interval = 1
        self.edge_index = self._create_edge_index()
        self.reset()

    def _create_edge_index(self):
        edge_index = []
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i != j:
                    edge_index.append([i, j])
        return torch.tensor(edge_index, dtype=torch.long).t().contiguous().to(self.device)

    def reset(self, num_customers=None):
        if num_customers is not None:
            self.num_customers = num_customers
        self.rng = random.Random(self.seed)
        self.np_rng = np.random.RandomState(self.seed)

        self.nodes = list(range(self.num_nodes))

        center_x, center_y = self.grid_size // 2, self.grid_size // 2
        center_node = center_y * self.grid_size + center_x

        if center_node != 0:
            self.nodes[0], self.nodes[center_node] = self.nodes[center_node], self.nodes[0]

        self.depot = 0
        self.node_to_coord = {
            node: (idx % self.grid_size, idx // self.grid_size)
            for idx, node in enumerate(self.nodes)
        }

        available_nodes = self.nodes[1:]
        self.num_customers = min(self.num_customers, len(available_nodes))
        self.customer_nodes = self.rng.sample(available_nodes, self.num_customers)

        # Initialize expected demands and actual demands (actual demands are None until revealed)
        self.expected_demands = {node: self.rng.randint(1, 5) for node in self.customer_nodes}
        self.actual_demands = {node: None for node in self.customer_nodes}  # Actual demands revealed on visit
        self.visited = set()
        self.vehicle_load = self.vehicle_capacity
        self.current_node = self.depot

        self.baseline_distance_matrix = np.zeros((self.num_nodes, self.num_nodes))
        for i in range(self.num_nodes):
            xi, yi = i % self.grid_size, i // self.grid_size
            for j in range(self.num_nodes):
                xj, yj = j % self.grid_size, j // self.grid_size
                dist = abs(xi - xj) + abs(yi - yj)
                self.baseline_distance_matrix[i][j] = dist
        self.current_traffic_matrix = self.baseline_distance_matrix.copy()
        self.time_step = 0
        self.distance_matrix = self._generate_traffic_matrix()

        self.total_cost = 0
        self.step_count = 0
        return self._get_state()

    def _get_state(self):
        node_features = []
        for i in range(self.num_nodes):
            # Use expected demand if actual demand is not yet revealed, else use actual
            demand = (self.actual_demands[i] if self.actual_demands[i] is not None else self.expected_demands.get(i, 0)) if i in self.customer_nodes else 0
            # Include expected demand as a feature to inform the policy
            expected_demand = self.expected_demands.get(i, 0) if i in self.customer_nodes else 0
            visited = 1 if i in self.visited else 0
            is_current = 1 if i == self.current_node else 0
            vehicle_load = self.vehicle_load
            dist_from_current = self.distance_matrix[self.current_node][i]
            node_features.append([demand, expected_demand, visited, is_current, vehicle_load, dist_from_current])

        x = torch.tensor(node_features, dtype=torch.float, device=self.device)

        src, tgt = self.edge_index
        edge_attr = torch.tensor(
            self.distance_matrix[src.cpu(), tgt.cpu()],
            dtype=torch.float32, device=self.device
        ).unsqueeze(1)

        return Data(x=x, edge_index=self.edge_index, edge_attr=edge_attr).to(self.device)

    def _generate_traffic_matrix(self):
        beta = self.traffic_transition_beta
        noise_std = self.traffic_noise_std
        current_traffic = torch.tensor(self.current_traffic_matrix, dtype=torch.float, device=self.device)
        base_distance = torch.tensor(self.baseline_distance_matrix, dtype=torch.float, device=self.device)
        noise = torch.normal(mean=0.0, std=noise_std, size=current_traffic.shape, device=self.device)
        updated_cost = beta * current_traffic + (1 - beta) * base_distance + noise + 5
        updated_cost = torch.max(base_distance, updated_cost)
        updated_cost.fill_diagonal_(0)
        self.current_traffic_matrix = updated_cost.cpu().numpy()
        self.time_step += 1
        return self.current_traffic_matrix



    def step(self, action):
        self.step_count += 1
        if self.step_count % self.traffic_update_interval == 0:
            self.distance_matrix = self._generate_traffic_matrix()

        cost = self.distance_matrix[self.current_node][action]
        self.total_cost += cost
        self.current_node = action
        done = False

        if action in self.customer_nodes and action not in self.visited:
            # Reveal actual demand upon visiting
            if self.actual_demands[action] is None:
                # Sample actual demand from a normal distribution around expected demand
                expected_demand = self.expected_demands[action]
                actual_demand = max(1, int(np.round(self.np_rng.normal(expected_demand, self.demand_noise_std))))
                # Ensure actual demand does not exceed vehicle capacity to avoid infeasibility
                actual_demand = min(actual_demand, self.vehicle_capacity - 1)
                self.actual_demands[action] = actual_demand
            demand = self.actual_demands[action]
            if self.vehicle_load >= demand:
                self.vehicle_load -= demand
                self.visited.add(action)

        if action == self.depot:
            self.vehicle_load = self.vehicle_capacity

        if len(self.visited) == len(self.customer_nodes):
            if self.current_node == self.depot:
                done = True

        return self._get_state(), -cost, done

    def save_state(self):
        return {
            'nodes': self.nodes[:],
            'customer_nodes': self.customer_nodes[:],
            'expected_demands': self.expected_demands.copy(),
            'actual_demands': self.actual_demands.copy(),
            'visited': self.visited.copy(),
            'vehicle_load': self.vehicle_load,
            'current_node': self.current_node,
            'total_cost': self.total_cost,
            'step_count': self.step_count,
            'time_step': self.time_step,
            'current_traffic_matrix': self.current_traffic_matrix.copy(),
            'rng_state': self.rng.getstate(),
            'np_rng_state': self.np_rng.get_state()
        }

    def load_state(self, state):
        self.nodes = state['nodes'][:]
        self.customer_nodes = state['customer_nodes'][:]
        self.expected_demands = state['expected_demands'].copy()
        self.actual_demands = state['actual_demands'].copy()
        self.visited = state['visited'].copy()
        self.vehicle_load = state['vehicle_load']
        self.current_node = state['current_node']
        self.total_cost = state['total_cost']
        self.step_count = state['step_count']
        self.time_step = state['time_step']
        self.current_traffic_matrix = state['current_traffic_matrix'].copy()
        self.rng.setstate(state['rng_state'])
        self.np_rng.set_state(state['np_rng_state'])

In [ ]:
class GNNPolicy(nn.Module):

    def __init__(self, input_dim=6, hidden_dim=64, num_heads=4):
        super(GNNPolicy, self).__init__()

        self.edge_nn1 = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim * hidden_dim)
        )
        self.edge_nn2 = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, hidden_dim * hidden_dim)
        )

        self.conv1 = NNConv(input_dim, hidden_dim, self.edge_nn1, aggr='mean')
        self.conv2 = NNConv(hidden_dim, hidden_dim, self.edge_nn2, aggr='mean')

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=num_heads,
                dim_feedforward=hidden_dim * 2,
                batch_first=True
            ),
            num_layers=2
        )

        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.hidden_state = None

        self.fc = nn.Linear(hidden_dim, 1)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def reset_memory(self):
        self.hidden_state = None

    def forward(self, data):
        x = data.x.to(device)
        edge_index = data.edge_index.to(device)
        edge_attr = data.edge_attr.to(device)

        if hasattr(data, 'batch') and data.batch is not None:
            batch = data.batch.to(device)
        else:
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)

        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.relu(self.conv2(x, edge_index, edge_attr))

        x = x.unsqueeze(0)
        x = self.transformer(x)
        x = x.squeeze(0)

        pooled = global_mean_pool(x, batch).unsqueeze(0)

        if self.hidden_state is None:
            h0 = torch.zeros(1, pooled.size(1), self.gru.hidden_size, device=device)
        else:
            h0 = self.hidden_state.to(device)

        _, self.hidden_state = self.gru(pooled, h0)
        self.hidden_state = self.hidden_state.detach()

        pooled_squeezed = pooled.squeeze(0)
        pooled_per_node = pooled_squeezed[batch]

        combined = x + pooled_per_node

        logits = self.fc(combined).squeeze(-1)

        out = F.softmax(logits, dim=-1) + 1e-10
        return out


# --- train_ppo Function with Changes ---
import torch
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

def train_ppo(iter=100, gamma=0.5, clip_eps=0.2, ppo_epochs=4, num_customers=50, policy=None, optimizer=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    grid_size = 8

    max_possible_customers = grid_size * grid_size - 1
    num_customers = min(num_customers, max_possible_customers)

    env = CVRPGridEnv(grid_size=grid_size, vehicle_capacity=15, num_customers=num_customers, device=device)

    if policy is None:
        policy = GNNPolicy(input_dim=6).to(device)

    if optimizer is None:
        optimizer = torch.optim.Adam(policy.parameters(), lr=1e-4)
        max_grad_norm = 1.0

    max_steps = env.num_nodes * 5

    losses = []
    returns_history = []

    for episode in range(iter):
        policy.reset_memory()
        num_cust = np.random.randint(10, max_possible_customers + 1)
        num_cust = min(num_cust, num_customers if num_customers > 0 else max_possible_customers)
        if num_cust == 0: num_cust = 1

        state = env.reset(num_customers=num_cust).to(device)

        episode_loss = 0.0
        episode_return = 0.0

        states = []
        actions = []
        log_probs_old = []
        rewards = []
        masks = []
        done = False
        step = 0

        while not done and step < max_steps:
            probs = policy(state)
            mask = torch.tensor([
                1.0 if (
                    (i == env.depot) or
                    (i in env.customer_nodes and i not in env.visited and env.expected_demands[i] <= env.vehicle_load)
                ) else 0.0 for i in range(env.num_nodes)
            ], device=probs.device)

            masked_probs = probs * mask
            if masked_probs.sum() == 0:
                if env.current_node != env.depot and env.depot in [i for i in range(env.num_nodes) if mask[i] > 0]:
                    action = torch.tensor(env.depot, device=device)
                    depot_probs = torch.zeros_like(probs)
                    depot_probs[env.depot] = 1.0
                    dist = torch.distributions.Categorical(depot_probs)
                    log_prob = dist.log_prob(action)
                else:
                    print(f"Warning: Masked probabilities sum to zero at episode {episode}, step {step}. Mask: {mask}")
                    valid_indices = torch.where(mask > 0)[0]
                    if valid_indices.numel() > 0:
                        uniform_probs = torch.zeros_like(probs).scatter_(0, valid_indices, 1.0 / valid_indices.numel())
                        dist = torch.distributions.Categorical(uniform_probs)
                        action = dist.sample()
                        log_prob = dist.log_prob(action)
                    else:
                        action = torch.tensor(env.depot, device=device)
                        depot_probs = torch.zeros_like(probs)
                        depot_probs[env.depot] = 1.0
                        dist = torch.distributions.Categorical(depot_probs)
                        log_prob = dist.log_prob(action)
            else:
                masked_probs /= masked_probs.sum()
                if torch.isnan(masked_probs).any():
                    print(f"NaNs detected in masked_probs at episode {episode}, step {step}")
                    action = torch.tensor(env.depot, device=device)
                    dist = torch.distributions.Categorical(torch.ones_like(probs) / probs.numel())
                    log_prob = dist.log_prob(action)
                else:
                    dist = torch.distributions.Categorical(masked_probs)
                    action = dist.sample()
                    log_prob = dist.log_prob(action)

            states.append(state)
            actions.append(action)
            log_probs_old.append(log_prob)
            masks.append(mask)

            prev_cost = env.total_cost
            prev_visited = len(env.visited)

            state, env_reward, done = env.step(action.item())

            delta_cost = prev_cost - env.total_cost
            delta_visited = len(env.visited) - prev_visited

            reward = 0.0
            reward += delta_cost
            reward += 5.0 * delta_visited
            reward -= 0.5

            rewards.append(reward)
            step += 1

        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)

        returns = torch.tensor(returns, device=device, dtype=torch.float32)

        if returns.numel() == 0:
            print(f"Episode {episode:04d} ended with no steps. Skipping PPO update.")
            losses.append(0.0)
            returns_history.append(0.0)
            continue

        baseline = returns.mean()
        advantages = returns - baseline

        if torch.isnan(advantages).any() or torch.isinf(advantages).any():
            print(f"NaNs or Infs detected in advantages at episode {episode}. Skipping PPO update.")
            losses.append(episode_loss / max(1, ppo_epochs))
            returns_history.append(sum(rewards) + (-10.0 * (env.num_customers - len(env.visited))))
            continue

        actions = torch.stack(actions)
        log_probs_old = torch.stack(log_probs_old).detach()
        advantages = advantages.detach()

        for ppo_epoch in range(ppo_epochs):
            log_probs_new = []
            entropies = []

            for t in range(len(states)):
                state_t = states[t]
                mask_t = masks[t]

                probs = policy(state_t)
                masked_probs = probs * mask_t

                if masked_probs.sum() == 0:
                    valid_indices = torch.where(mask_t > 0)[0]
                    if valid_indices.numel() > 0:
                        uniform_probs = torch.zeros_like(probs).scatter_(0, valid_indices, 1.0 / valid_indices.numel())
                        dist = torch.distributions.Categorical(uniform_probs)
                    else:
                        dist = torch.distributions.Categorical(torch.ones_like(probs) / probs.numel())
                else:
                    masked_probs /= masked_probs.sum()
                    dist = torch.distributions.Categorical(masked_probs)

                log_prob = dist.log_prob(actions[t])
                entropy = dist.entropy()

                log_probs_new.append(log_prob)
                entropies.append(entropy)

            if not log_probs_new:
                print(f"Skipping PPO update epoch {ppo_epoch} for episode {episode}: No valid log probabilities calculated.")
                continue

            log_probs_new = torch.stack(log_probs_new)
            entropies = torch.stack(entropies)

            ratios = torch.exp(log_probs_new - log_probs_old)
            clipped_ratios = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps)
            surrogate1 = ratios * advantages
            surrogate2 = clipped_ratios * advantages

            policy_loss = -torch.min(surrogate1, surrogate2).mean()
            entropy_bonus = entropies.mean()
            loss = policy_loss - 0.01 * entropy_bonus

            optimizer.zero_grad()
            loss.backward()

            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)

            optimizer.step()

            episode_loss += loss.item()

        episode_return = sum(rewards) + (-10.0 * (env.num_customers - len(env.visited)))

        losses.append(episode_loss / max(1, ppo_epochs))
        returns_history.append(episode_return)

        if episode % 100 == 0:
            print(f"Episode {episode:04d} | Loss: {losses[-1]:.4f} | Return: {returns_history[-1]:.2f} | Visited: {len(env.visited)}/{env.num_customers}")

    return policy, losses, returns_history


# Run PPO training
# Keep the number of customers reasonable for the initial run
policy, losses, returns = train_ppo(iter=11, num_customers=30)
torch.save(policy.state_dict(), "gnn_policy_ppo.pth")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch

def evaluate_gnn_policy(policy, env):
    state = env.reset().to(device)
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 5
    step = 0

    while not done and step < max_steps:
        with torch.no_grad():
            probs = policy(state)
        mask = torch.tensor([
            1.0 if (
                (i == env.depot) or
                (i in env.customer_nodes and i not in env.visited and env.expected_demands[i] <= env.vehicle_load)
            ) else 0.0 for i in range(env.num_nodes)
        ], device=probs.device)
        masked_probs = probs * mask
        if masked_probs.sum() == 0:
            action = env.depot
        else:
            masked_probs /= masked_probs.sum()
            action = torch.multinomial(masked_probs, 1).item()
        state, step_reward, done = env.step(action)
        route.append(action)
        step_costs.append(-step_reward)
        step += 1

        # Check feasibility
        feasible_remaining = any(
            node not in env.visited and env.expected_demands[node] <= env.vehicle_load
            for node in env.customer_nodes
        )
        if not done and (not feasible_remaining or env.vehicle_load == 0):
            if env.current_node != env.depot:
                state, step_reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-step_reward)
            # Don't break, let the loop continue to allow depot returns

    total_cost = sum(step_costs)
    return total_cost, len(env.visited), route, step_costs

In [ ]:
def traffic_aware_tabu_search(env, max_iterations=500, tabu_tenure=20):
    env.reset()
    route = [env.depot]
    step_costs = []
    tabu_list = []
    iteration = 0
    max_steps = env.num_nodes * 50  # Safety bound

    while len(env.visited) < len(env.customer_nodes) and iteration < max_iterations and iteration < max_steps:
        # 🔄 Update traffic matrix
        env.distance_matrix = env._generate_traffic_matrix()
        current = env.current_node

        if current == env.depot:
            # Refill when at depot
            env.vehicle_load = env.vehicle_capacity

        # Get all unvisited customers
        unvisited_customers = [
            node for node in env.customer_nodes if node not in env.visited
        ]

        if not unvisited_customers:
            break

        # Choose next customer by distance, considering tabu
        distances = [(node, env.distance_matrix[current][node]) for node in unvisited_customers]
        distances.sort(key=lambda x: x[1])

        for node, dist in distances:
            if node not in tabu_list or len(distances) == 1:
                action = node
                break
        else:
            action = distances[0][0]  # All tabu, pick best anyway

        # Execute move to customer
        _, reward, _ = env.step(action)
        route.append(action)
        step_costs.append(-reward)
        iteration += 1

        # If the visit failed (demand > load), it won’t be marked visited
        if action in env.visited:
            # Only successful visits go into tabu list
            tabu_list.append(action)
            if len(tabu_list) > tabu_tenure:
                tabu_list.pop(0)
            continue

        # Delivery failed — return to depot to refill
        if env.current_node != env.depot:
            _, reward, _ = env.step(env.depot)
            route.append(env.depot)
            step_costs.append(-reward)
            iteration += 1

    # Ensure return to depot
    if route[-1] != env.depot:
        last = route[-1]
        step_costs.append(env.current_traffic_matrix[last][env.depot])
        route.append(env.depot)

    total_cost = sum(step_costs)
    return total_cost, len(env.visited), route, step_costs


In [ ]:
import torch

def evaluate_hybrid_policy(policy, env, k=3):
    state = env.reset().to(device)
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 5
    step = 0

    while not done and step < max_steps:
        with torch.no_grad():
            probs = policy(state).clone()

        current = env.current_node
        feasible_nodes = [
            node for node in env.customer_nodes
            if node not in env.visited and env.expected_demands[node] <= env.vehicle_load
        ]

        if not feasible_nodes:
            action = env.depot
        else:
            distances = [(node, env.distance_matrix[current][node]) for node in feasible_nodes]
            top_k_nodes = [node for node, _ in sorted(distances, key=lambda x: x[1])[:k]]

            greedy_mask = torch.tensor([
                1.0 if i in top_k_nodes or i == env.depot else 0.0
                for i in range(env.num_nodes)
            ], device=probs.device)

            masked_probs = probs * greedy_mask
            if masked_probs.sum() == 0:
                action = env.depot
            else:
                masked_probs /= masked_probs.sum()
                action = torch.multinomial(masked_probs, 1).item()

        state, step_reward, done = env.step(action)
        route.append(action)
        step_costs.append(-step_reward)
        step += 1

        feasible_remaining = any(
            node not in env.visited and env.expected_demands[node] <= env.vehicle_load
            for node in env.customer_nodes
        )
        if not done and (not feasible_remaining or env.vehicle_load == 0):
            if env.current_node != env.depot:
                state, step_reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-step_reward)

    # Ensure route ends with depot
    if route[-1] != env.depot:
        last = route[-1]
        step_costs.append(env.current_traffic_matrix[last][env.depot])
        route.append(env.depot)

    total_cost = sum(step_costs)
    return total_cost, len(env.visited), route, step_costs

In [ ]:
def greedy_search_traffic_aware(env, return_route=True):
    env.reset()
    route = [env.depot]
    step_costs = []
    max_steps = env.num_nodes * 50
    step = 0

    while len(env.visited) < len(env.customer_nodes) and step < max_steps:
        # Update traffic matrix
        env.distance_matrix = env._generate_traffic_matrix()
        current = env.current_node

        if current == env.depot:
            # Refill at depot
            env.vehicle_load = env.vehicle_capacity

        # Try to find a feasible customer based on current load
        unvisited_customers = [
            node for node in env.customer_nodes if node not in env.visited
        ]

        if not unvisited_customers:
            break

        # Always evaluate distance to all unvisited customers
        distances = [(node, env.distance_matrix[current][node]) for node in unvisited_customers]
        action = min(distances, key=lambda x: x[1])[0]

        # Take the step (even if not feasible), record cost
        _, reward, _ = env.step(action)
        route.append(action)
        step_costs.append(-reward)
        step += 1

        # Check if delivery was successful
        if action in env.visited:
            continue  # Delivery succeeded

        # Delivery failed: insufficient load, return to depot
        if env.current_node != env.depot:
            # Go back to depot
            _, reward, _ = env.step(env.depot)
            route.append(env.depot)
            step_costs.append(-reward)
            step += 1
            # On next loop, it will refill and try again

    # Ensure route ends at depot
    if route[-1] != env.depot:
        last = route[-1]
        step_costs.append(env.current_traffic_matrix[last][env.depot])
        route.append(env.depot)

    total_cost = sum(step_costs)
    if return_route:
        return total_cost, len(env.visited), route, step_costs
    return total_cost, len(env.visited)


In [ ]:
import random

def random_search(env, return_route=True):
    env.reset()
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 10  # increased for extra depot trips
    step = 0

    while not done and step < max_steps:
        env.distance_matrix = env._generate_traffic_matrix()
        current = env.current_node

        # Use actual demands if present, fallback to expected demands
        demands = getattr(env, 'actual_demands', env.expected_demands)

        # Filter customers not visited and whose demand fits current load
        feasible_nodes = [
            node for node in env.customer_nodes
            if node not in env.visited and (demands.get(node) or env.expected_demands.get(node, 0)) <= env.vehicle_load
        ]

        if not feasible_nodes:
            # No feasible customers, return to depot if not already there
            if env.current_node != env.depot:
                action = env.depot
            else:
                # Already at depot and no feasible customers - end
                break
        else:
            action = random.choice(feasible_nodes)

        # Before visiting a customer, if demand exceeds current load, return to depot first
        demand = demands.get(action) or env.expected_demands.get(action, 0)
        if action != env.depot and demand > env.vehicle_load:
            if env.current_node != env.depot:
                # Go back to depot first to refill
                state, reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-reward)
                env.vehicle_load = env.vehicle_capacity
                current = env.depot
                step += 1
                if done:
                    break
            else:
                # Already at depot but demand still too high (maybe demand > capacity)
                # Proceed anyway (or could break)
                pass

        if done:
            break

        # Take a step to chosen action
        state, reward, done = env.step(action)
        route.append(action)
        step_costs.append(-reward)
        step += 1

    # Ensure route ends at depot
    if route[-1] != env.depot:
        last = route[-1]
        cost_to_depot = env.current_traffic_matrix[last][env.depot]
        step_costs.append(cost_to_depot)
        route.append(env.depot)

    total_cost = sum(step_costs)
    if return_route:
        return total_cost, len(env.visited), route, step_costs
    return total_cost, len(env.visited)


In [ ]:
import random

def random_search(env, return_route=True):
    env.reset()
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 50  # Match greedy's step limit for fair comparison
    step = 0

    while not done and step < max_steps:
        env.distance_matrix = env._generate_traffic_matrix()
        current = env.current_node

        # Filter only unvisited customers (no demand feasibility check)
        unvisited_customers = [
            node for node in env.customer_nodes if node not in env.visited
        ]

        if not unvisited_customers:
            # No unvisited customers left, return to depot if not there
            if env.current_node != env.depot:
                action = env.depot
            else:
                # Already at depot with no customers to visit - end
                done = True
                break
        else:
            # Randomly select an unvisited customer
            action = random.choice(unvisited_customers)

        # Take a step to the chosen action
        state, reward, done = env.step(action)
        route.append(action)
        step_costs.append(-reward)
        step += 1

        # Check if delivery was successful (based on actual demand)
        if action in env.visited:
            continue  # Delivery succeeded, proceed to next step

        # Delivery failed: insufficient load, return to depot if not already there
        if action != env.depot and env.current_node != env.depot:
            state, reward, done = env.step(env.depot)
            route.append(env.depot)
            step_costs.append(-reward)
            step += 1
            # On next loop, vehicle will refill at depot (env.step handles refill)

        if done:
            break

    # Ensure route ends at depot
    if route[-1] != env.depot:
        last = route[-1]
        cost_to_depot = env.current_traffic_matrix[last][env.depot]
        step_costs.append(cost_to_depot)
        route.append(env.depot)

    total_cost = sum(step_costs)
    if return_route:
        return total_cost, len(env.visited), route, step_costs
    return total_cost, len(env.visited)

In [ ]:
def largest_demand_search(env, return_route=True):
    env.reset()
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 10  # Increased for extra depot trips
    step = 0

    while not done and step < max_steps:
        env.distance_matrix = env._generate_traffic_matrix()
        current = env.current_node

        # Use actual demands if available, else fallback to expected demands
        # The actual_demands dict exists from env.reset, but values are None until visited.
        demands = env.actual_demands # Use actual_demands if it exists

        # Filter feasible nodes: unvisited and demand fits current load
        feasible_nodes = [
            node for node in env.customer_nodes
            # Use expected demand if actual demand is None (not yet revealed)
            if node not in env.visited and (demands.get(node) if demands.get(node) is not None else env.expected_demands.get(node, 0)) <= env.vehicle_load
        ]

        if not feasible_nodes:
            # No feasible nodes: return to depot if not there already
            if env.current_node != env.depot:
                action = env.depot
            else:
                break  # At depot and no feasible customers means done
        else:
            # Pick the node with largest demand among feasible nodes
            # Use actual demand if revealed, otherwise use expected demand for comparison
            action = max(feasible_nodes, key=lambda node: demands.get(node) if demands.get(node) is not None else env.expected_demands.get(node, 0))

        # If chosen customer demand exceeds current load, go to depot first
        # Use actual demand for the check if revealed, else expected
        demand_to_check = demands.get(action) if demands.get(action) is not None else env.expected_demands.get(action, 0)

        if action != env.depot and demand_to_check > env.vehicle_load:
            if env.current_node != env.depot:
                # Go to depot to refill first
                state, reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-reward)
                env.vehicle_load = env.vehicle_capacity
                current_node = env.depot
                step += 1
                if done:
                    break
            else:
                # Already at depot but demand still too large (maybe demand > capacity)
                # Proceed anyway (or could break)
                pass # This case should ideally not happen if capacity > max demand

        if done:
            break

        # Take step to chosen node
        # The env.step method handles revealing actual demand and updating visited/vehicle_load
        state, reward, done = env.step(action)
        route.append(action)
        step_costs.append(-reward)
        current_node = action # env.step already updates env.current_node, but update local variable for clarity
        step += 1


    # End route at depot if not already
    if route and route[-1] != env.depot:
        last = route[-1]
        # Need to get the cost from the *current* traffic matrix
        # Ensure the matrix is updated one last time if needed
        env.distance_matrix = env._generate_traffic_matrix()
        cost_to_depot = env.distance_matrix[last][env.depot]
        step_costs.append(cost_to_depot)
        route.append(env.depot)


    total_cost = sum(step_costs)
    # Ensure total cost is calculated correctly from accumulated step_costs
    if return_route:
        # The route starts with depot, append it here
        final_route = [env.depot] + route[1:] if route and route[0] != env.depot else route
        return total_cost, len(env.visited), final_route, step_costs
    return total_cost, len(env.visited)

In [ ]:
def max_reachable_search(env, return_route=True):
    env.reset()
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 10  # Increased to allow depot returns
    step = 0

    while not done and step < max_steps:
        env.distance_matrix = env._generate_traffic_matrix()
        current = env.current_node

        # Use actual demands if available, else fallback to expected demands
        # The actual_demands dict exists, but values are None until visited.
        # Need to check if the VALUE in actual_demands is None.
        # demands = getattr(env, 'actual_demands', env.expected_demands) # Original line

        # Filter feasible nodes: unvisited and demand fits current load
        feasible_nodes = [
            node for node in env.customer_nodes
            # Check if actual demand is None; if so, use expected demand. Otherwise, use actual demand.
            if node not in env.visited and
               (env.actual_demands.get(node) if env.actual_demands.get(node) is not None else env.expected_demands.get(node, 0)) <= env.vehicle_load
        ]

        if not feasible_nodes:
            # No feasible nodes, return to depot if not already there
            if env.current_node != env.depot:
                action = env.depot
            else:
                break  # Done if at depot with no feasible nodes
        else:
            def count_reachable(node):
                # Use the same logic for getting the node's demand
                node_demand = env.actual_demands.get(node) if env.actual_demands.get(node) is not None else env.expected_demands.get(node, 0)
                remaining_capacity = env.vehicle_load - node_demand
                reachable = sum(
                    1 for other_node in env.customer_nodes
                    if other_node not in env.visited and other_node != node and
                       (env.actual_demands.get(other_node) if env.actual_demands.get(other_node) is not None else env.expected_demands.get(other_node, 0)) <= remaining_capacity
                )
                return reachable

            action = max(feasible_nodes, key=count_reachable)

        # If action is a customer and demand > current load, go to depot first
        # Use the correct logic for getting the demand here too
        demand = env.actual_demands.get(action) if env.actual_demands.get(action) is not None else env.expected_demands.get(action, 0)

        if action != env.depot and demand > env.vehicle_load:
            if env.current_node != env.depot:
                # Go to depot to refill
                state, reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-reward)
                env.vehicle_load = env.vehicle_capacity
                current = env.depot
                step += 1
                if done:
                    break

        if done:
            break

        # Take step to chosen node
        state, reward, done = env.step(action)
        route.append(action)
        step_costs.append(-reward)
        step += 1

    # End route at depot if not already
    if route[-1] != env.depot:
        last = route[-1]
        cost_to_depot = env.current_traffic_matrix[last][env.depot]
        step_costs.append(cost_to_depot)
        route.append(env.depot)

    total_cost = sum(step_costs)
    if return_route:
        return total_cost, len(env.visited), route, step_costs
    return total_cost, len(env.visited)

In [ ]:
def clarke_wright(env, return_route=True):
    env.reset()
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 10  # Allow extra steps for depot returns
    step = 0

    # Force traffic update at the start
    env.distance_matrix = env._generate_traffic_matrix()
    traffic_matrix = env.current_traffic_matrix

    depot = env.depot
    customers = env.customer_nodes

    # Use expected_demands if present, else fallback to demands
    demands = getattr(env, 'expected_demands', env.expected_demands)
    vehicle_capacity = env.vehicle_capacity

    # Step 1: Compute savings for all customer pairs
    savings = []
    for i in customers:
        for j in customers:
            if i < j:
                s_ij = traffic_matrix[depot][i] + traffic_matrix[depot][j] - traffic_matrix[i][j]
                savings.append((s_ij, i, j))
    savings.sort(reverse=True)

    # Step 2: Initialize routes (each customer own route)
    routes = [[depot, i, depot] for i in customers]

    # Step 3: Merge routes based on savings and capacity feasibility
    while savings and step < max_steps:
        s_ij, i, j = savings.pop(0)

        route_i = None
        route_j = None
        for r in routes:
            if i in r:
                route_i = r
            if j in r:
                route_j = r
            if route_i and route_j:
                break
        if route_i is None or route_j is None or route_i == route_j:
            continue

        if (route_i[-2] == i and route_j[1] == j):
            total_demand = sum(demands[node] for node in route_i[1:-1] + route_j[1:-1])
            if total_demand <= vehicle_capacity:
                new_route = route_i[:-1] + route_j[1:]
                routes.remove(route_i)
                routes.remove(route_j)
                routes.append(new_route)
        elif (route_j[-2] == j and route_i[1] == i):
            total_demand = sum(demands[node] for node in route_j[1:-1] + route_i[1:-1])
            if total_demand <= vehicle_capacity:
                new_route = route_j[:-1] + route_i[1:]
                routes.remove(route_i)
                routes.remove(route_j)
                routes.append(new_route)

        step += 1

    # Step 4: Filter feasible routes (using expected_demands)
    feasible_routes = [r for r in routes if sum(demands[node] for node in r[1:-1]) <= vehicle_capacity]
    if not feasible_routes:
        feasible_routes = [[depot, depot]]

    # Flatten routes for simulation, keep depot at route starts only
    cw_route = []
    for r in feasible_routes:
        for idx, node in enumerate(r):
            if node != depot or idx == 0:
                cw_route.append(node)
    if cw_route[-1] != depot:
        cw_route.append(depot)

    # Step 5: Simulate route in environment with actual demands & uncertainty
    route = []
    env.vehicle_load = vehicle_capacity
    env.visited = set()
    current_node = depot
    step = 0
    done = False

    for node in cw_route[1:]:
        if step >= max_steps or done:
            break

        if node == depot:
            if route and route[-1] != depot:
                state, reward, done = env.step(depot)
                route.append(depot)
                step_costs.append(-reward)
                env.vehicle_load = vehicle_capacity
                current_node = depot
                step += 1
            continue

        # Check actual demand before serving
        actual_demand = env.actual_demands.get(node) or demands.get(node) or 0

        # If actual demand > current load, return to depot to refill first
        while actual_demand > env.vehicle_load and step < max_steps:
            if current_node != depot:
                state, reward, done = env.step(depot)
                route.append(depot)
                step_costs.append(-reward)
                env.vehicle_load = vehicle_capacity
                current_node = depot
                step += 1
                if done:
                    break
            else:
                # Already at depot but demand still too large? Assume refill fixed it.
                break

        if done:
            break

        # Visit customer if not visited and demand fits load
        if node not in env.visited and actual_demand <= env.vehicle_load:
            state, reward, done = env.step(node)
            route.append(node)
            step_costs.append(-reward)
            current_node = node
            step += 1

    # Step 6: Visit missed customers individually if any remain unvisited
    for customer in customers:
        if customer not in env.visited and not done and step < max_steps:
            if current_node != depot:
                state, reward, done = env.step(depot)
                route.append(depot)
                step_costs.append(-reward)
                env.vehicle_load = vehicle_capacity
                current_node = depot
                step += 1

            actual_demand = env.actual_demands.get(customer) or demands.get(customer) or 0

            # If actual demand > capacity (shouldn't happen), forcibly reset load
            if actual_demand > env.vehicle_load:
                env.vehicle_load = vehicle_capacity

            state, reward, done = env.step(customer)
            route.append(customer)
            step_costs.append(-reward)
            current_node = customer
            step += 1

            if current_node != depot and step < max_steps:
                state, reward, done = env.step(depot)
                route.append(depot)
                step_costs.append(-reward)
                env.vehicle_load = vehicle_capacity
                current_node = depot
                step += 1

            if done:
                break

    # Ensure route ends at depot
    if route and route[-1] != depot and step < max_steps:
        last = route[-1]
        step_costs.append(traffic_matrix[last][depot])
        route.append(depot)

    total_cost = sum(step_costs)
    if return_route:
        return total_cost, len(env.visited), [depot] + route, step_costs
    return total_cost, len(env.visited)


In [ ]:
import random
import numpy as np

def sweep(env, return_route=True):
    env.reset()
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 50
    step = 0
    vehicle_capacity = env.vehicle_capacity  # Use vehicle_capacity for max load

    env.distance_matrix = env._generate_traffic_matrix()
    traffic_matrix = env.current_traffic_matrix

    customers = env.customer_nodes
    depot_coords = env.node_to_coord[env.depot]

    # Sort customers by angle relative to depot (sweep order)
    sorted_customers = sorted(
        customers,
        key=lambda node: np.arctan2(
            env.node_to_coord[node][1] - depot_coords[1],
            env.node_to_coord[node][0] - depot_coords[0]
        )
    )

    demands = env.expected_demands
    actual_demands = getattr(env, 'actual_demands', demands)

    sweep_routes = [[env.depot]]
    current_demand = 0
    remaining_customers = sorted_customers.copy()

    while remaining_customers:
        current_route = sweep_routes[-1]
        added = False
        for node in remaining_customers[:]:
            node_demand = demands.get(node, 0)
            if node_demand <= vehicle_capacity - current_demand:
                current_route.append(node)
                current_demand += node_demand
                remaining_customers.remove(node)
                added = True
        if not added or not remaining_customers:
            if len(current_route) > 1:
                current_route.append(env.depot)
                sweep_routes.append([env.depot])
                current_demand = 0
            if not remaining_customers:
                break
            node = remaining_customers.pop(0)
            sweep_routes.append([env.depot, node, env.depot])
            current_demand = 0

    if sweep_routes[-1][-1] != env.depot and len(sweep_routes[-1]) > 1:
        sweep_routes[-1].append(env.depot)
    sweep_routes = [r for r in sweep_routes if len(r) > 2 or r[-1] != env.depot]

    sweep_route = [node for route in sweep_routes for node in route]
    if not sweep_route or sweep_route == [env.depot]:
        sweep_route = [env.depot] + sorted_customers + [env.depot]
    elif sweep_route[-1] != env.depot:
        sweep_route.append(env.depot)

    route = []
    env.vehicle_load = vehicle_capacity
    env.visited = set()
    current_node = env.depot

    for node in sweep_route[1:]:
        if step >= max_steps or done:
            break

        if node == env.depot:
            if route and route[-1] != env.depot:
                state, reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-reward)
                env.vehicle_load = vehicle_capacity
                current_node = env.depot
                step += 1
            continue

        # Use actual demand at execution time with safe fallback
        demand = actual_demands.get(node) or demands.get(node) or 0

        # Return to depot if demand exceeds current load
        while demand > env.vehicle_load and step < max_steps:
            if current_node != env.depot:
                state, reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-reward)
                env.vehicle_load = vehicle_capacity
                current_node = env.depot
                step += 1
                if done:
                    break
            else:
                break  # already at depot, assume refill done

        if done:
            break

        if node not in env.visited and demand <= env.vehicle_load:
            state, reward, done = env.step(node)
            route.append(node)
            step_costs.append(-reward)
            env.vehicle_load = max(0, env.vehicle_load - demand)
            current_node = node
            step += 1

    # Visit any unvisited customers
    unvisited_customers = [node for node in customers if node not in env.visited]
    while unvisited_customers and step < max_steps and not done:
        node = unvisited_customers[0]
        demand = actual_demands.get(node) or demands.get(node) or 0

        while demand > env.vehicle_load and step < max_steps:
            state, reward, done = env.step(env.depot)
            route.append(env.depot)
            step_costs.append(-reward)
            env.vehicle_load = vehicle_capacity
            current_node = env.depot
            step += 1
            if done:
                break
        if done:
            break

        if node not in env.visited and demand <= env.vehicle_load:
            state, reward, done = env.step(node)
            route.append(node)
            step_costs.append(-reward)
            env.vehicle_load = max(0, env.vehicle_load - demand)
            current_node = node
            step += 1

        unvisited_customers = [n for n in customers if n not in env.visited]

    # Ensure route ends at depot
    if route and route[-1] != env.depot and step < max_steps:
        last = route[-1]
        step_costs.append(traffic_matrix[last][env.depot])
        route.append(env.depot)

    total_cost = sum(step_costs)
    if return_route:
        return total_cost, len(env.visited), [env.depot] + route, step_costs
    return total_cost, len(env.visited)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import os

def plot_routes(
    env,
    gnn_info,
    tabu_info,
    greedy_ta_info,
    hybrid_info,
    random_info,
    largest_info,
    max_reach_info,
    cw_info,
    sweep_info,
    hybrid_cw_info,
    env_index
):
    """Plot CVRP routes for 10 methods in a 2x5 grid with traffic-aware edge coloring."""
    fig, axes = plt.subplots(2, 5, figsize=(30, 12), sharex=True, sharey=True, constrained_layout=True)
    fig.suptitle(f'CVRP Routes - Environment {env_index + 1}', fontsize=20)

    methods = [
        'GNN Policy',
        'Tabu Search',
        'Greedy-Traffic Aware',
        'Hybrid (Top-k GNN + Greedy)',
        'Random',
        'Largest-Demand',
        'Max-Reachable',
        'Clarke-Wright',
        'Sweep'

    ]
    infos = [
        gnn_info,
        tabu_info,
        greedy_ta_info,
        hybrid_info,
        random_info,
        largest_info,
        max_reach_info,
        cw_info,
        sweep_info
    ]
    colors = ['blue', 'green', 'purple', 'orange', 'cyan', 'brown', 'magenta', 'red', 'lime']

    coords = env.node_to_coord
    max_demand = max(env.demands.values()) if env.demands else 1

    # Gather all traffic factors across all routes for normalization
    all_traffic_factors = []
    for route, _ in infos:
        for i in range(len(route) - 1):
            start, end = route[i], route[i + 1]
            base_dist = env.baseline_distance_matrix[start][end]
            current_dist = env.current_traffic_matrix[start][end]
            traffic_factor = current_dist / base_dist if base_dist > 0 else 1.0
            all_traffic_factors.append(traffic_factor)

    vmin = min(all_traffic_factors or [1.0])
    vmax = max(all_traffic_factors or [1.0])
    if vmin == vmax:
        vmax = vmin + 1e-3
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.colormaps.get_cmap('RdYlGn_r')

    for idx, (method, (route, step_costs), base_color) in enumerate(zip(methods, infos, colors)):
        ax = axes[idx // 5][idx % 5]
        ax.grid(True)
        ax.set_xticks(range(env.grid_size))
        ax.set_yticks(range(env.grid_size))
        ax.set_xlim(-0.5, env.grid_size - 0.5)
        ax.set_ylim(-0.5, env.grid_size - 0.5)
        ax.invert_yaxis()
        ax.set_title(method)

        # Plot depot
        depot_x, depot_y = coords[env.depot]
        ax.plot(depot_x, depot_y, 's', color='red', markersize=15, label='Depot')

        # Plot customers
        for idx2, node in enumerate(env.customer_nodes):
            x, y = coords[node]
            demand = env.demands[node]
            size = 50 + 100 * (demand / max_demand)
            label = 'Customer' if idx2 == 0 else None
            ax.scatter(x, y, s=size, color='black', alpha=0.6, label=label)
            ax.text(x + 0.2 * (idx2 % 2 - 0.5), y + 0.2 * (idx2 % 2 - 0.5), str(node), fontsize=8)

        # Draw edges with traffic-based coloring
        for i in range(len(route) - 1):
            start, end = route[i], route[i + 1]
            x1, y1 = coords[start]
            x2, y2 = coords[end]

            base_dist = env.baseline_distance_matrix[start][end]
            current_dist = env.current_traffic_matrix[start][end]
            traffic_factor = current_dist / base_dist if base_dist > 0 else 1.0

            color_edge = cmap(norm(traffic_factor))
            ax.plot([x1, x2], [y1, y2], color=color_edge, linewidth=2, zorder=1)

            mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
            ax.text(mid_x, mid_y, f"{current_dist:.1f}", fontsize=7, color='black',
                    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))

        # Plot route points
        xs, ys = zip(*[coords[n] for n in route])
        ax.scatter(xs, ys, color=base_color, s=30, zorder=2)
        ax.legend()

    # Add colorbar for traffic factors below the plots
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), orientation='horizontal', fraction=0.05, pad=0.06)
    cbar.set_label('Traffic Factor (Current Distance / Baseline Distance)')

    output_dir = "./output"
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f'{output_dir}/cvrp_routes_env_{env_index + 1}_10models.png')
    plt.show()

In [ ]:
import torch
import numpy as np
from copy import deepcopy

def compare_methods(examples=10):
    policy = GNNPolicy().to(device)
    policy.load_state_dict(torch.load("gnn_policy_ppo.pth"))
    policy.eval()

    test_envs = [CVRPGridEnv(grid_size=8, vehicle_capacity=15, num_customers=50, seed=i) for i in range(examples)]

    gnn_results = []
    hybrid_results = []
    traffic_aware_results = []
    greedy_traffic_results = []
    random_results = []
    largest_demand_results = []
    max_reachable_results = []
    cw_results = []
    sweep_results = []


    print("Evaluating and visualizing on test environments...\n")
    for i, env in enumerate(test_envs):
        # Existing methods
        gnn_cost, gnn_visited, gnn_route, gnn_step_costs = evaluate_gnn_policy(policy, deepcopy(env))
        gnn_results.append((gnn_cost, gnn_visited))
        print(f"Env {i+1} - GNN: Cost={gnn_cost:.2f}, Visited={gnn_visited}/{env.num_customers}")

        hybrid_cost, hybrid_visited, hybrid_route, hybrid_step_costs = evaluate_hybrid_policy(policy, deepcopy(env), k=3)
        hybrid_results.append((hybrid_cost, hybrid_visited))
        print(f"Env {i+1} - Hybrid GNN+Greedy: Cost={hybrid_cost:.2f}, Visited={hybrid_visited}/{env.num_customers}")

        traffic_cost, traffic_visited, traffic_route, traffic_step_costs = traffic_aware_tabu_search(deepcopy(env))
        traffic_aware_results.append((traffic_cost, traffic_visited))
        print(f"Env {i+1} - Traffic-Aware Tabu: Cost={traffic_cost:.2f}, Visited={traffic_visited}/{env.num_customers}")

        greedy_ta_cost, greedy_ta_visited, greedy_ta_route, greedy_ta_step_costs = greedy_search_traffic_aware(deepcopy(env), return_route=True)
        greedy_traffic_results.append((greedy_ta_cost, greedy_ta_visited))
        print(f"Env {i+1} - Greedy-Traffic: Cost={greedy_ta_cost:.2f}, Visited={greedy_ta_visited}/{env.num_customers}")

        random_cost, random_visited, random_route, random_step_costs = random_search(deepcopy(env), return_route=True)
        random_results.append((random_cost, random_visited))
        print(f"Env {i+1} - Random: Cost={random_cost:.2f}, Visited={random_visited}/{env.num_customers}")

        largest_cost, largest_visited, largest_route, largest_step_costs = largest_demand_search(deepcopy(env), return_route=True)
        largest_demand_results.append((largest_cost, largest_visited))
        print(f"Env {i+1} - Largest-Demand: Cost={largest_cost:.2f}, Visited={largest_visited}/{env.num_customers}")

        max_reach_cost, max_reach_visited, max_reach_route, max_reach_step_costs = max_reachable_search(deepcopy(env), return_route=True)
        max_reachable_results.append((max_reach_cost, max_reach_visited))
        print(f"Env {i+1} - Max-Reachable: Cost={max_reach_cost:.2f}, Visited={max_reach_visited}/{env.num_customers}")

        cw_cost, cw_visited, cw_route, cw_step_costs = clarke_wright(deepcopy(env), return_route=True)
        cw_results.append((cw_cost, cw_visited))
        print(f"Env {i+1} - Clarke-Wright: Cost={cw_cost:.2f}, Visited={cw_visited}/{env.num_customers}")

        sweep_cost, sweep_visited, sweep_route, sweep_step_costs = sweep(deepcopy(env), return_route=True)
        sweep_results.append((sweep_cost, sweep_visited))
        print(f"Env {i+1} - Sweep: Cost={sweep_cost:.2f}, Visited={sweep_visited}/{env.num_customers}")
        print("\n")



    def mean_se(values):
        values = np.array(values)
        mean = np.mean(values)
        se = np.std(values, ddof=1) / np.sqrt(len(values))
        return mean, se

    def summarize(name, results, env):
        costs = [r[0] for r in results]
        visited = [r[1] / env.num_customers * 100 for r in results]

        mean_cost, se_cost = mean_se(costs)
        mean_visited, se_visited = mean_se(visited)

        print(f"{name:<20} Avg Cost = {mean_cost:.2f} ± {se_cost:.2f}, Avg Visited = {mean_visited:.2f}% ± {se_visited:.2f}%")

    print("\n=== Performance Summary ===")
    summarize("GNN Policy", gnn_results, env)
    summarize("Hybrid GNN+Greedy", hybrid_results, env)
    summarize("Traffic-Aware Tabu", traffic_aware_results, env)
    summarize("Greedy-Traffic", greedy_traffic_results, env)
    summarize("Random", random_results, env)
    summarize("Largest-Demand", largest_demand_results, env)
    summarize("Max-Reachable", max_reachable_results, env)
    summarize("Clarke-Wright", cw_results, env)
    summarize("Sweep", sweep_results, env)


if __name__ == "__main__":
    compare_methods(examples=30)

/var/folders/s1/pf0brh214k58csbxb6bc_w980000gn/T/ipykernel_7767/1152600484.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  policy.load_state_dict(torch.load("gnn_policy_

Evaluating and visualizing on test environments...

Env 1 - GNN: Cost=1079.26, Visited=50/50
Env 1 - Hybrid GNN+Greedy: Cost=1047.84, Visited=50/50
Env 1 - Traffic-Aware Tabu: Cost=1006.00, Visited=50/50
Env 1 - Greedy-Traffic: Cost=1034.92, Visited=50/50
Env 1 - Random: Cost=1151.81, Visited=50/50
Env 1 - Largest-Demand: Cost=1149.72, Visited=50/50
Env 1 - Max-Reachable: Cost=1132.39, Visited=50/50
Env 1 - Clarke-Wright: Cost=1012.50, Visited=50/50
Env 1 - Sweep: Cost=1253.44, Visited=50/50


Env 2 - GNN: Cost=1136.06, Visited=50/50
Env 2 - Hybrid GNN+Greedy: Cost=1066.55, Visited=50/50
Env 2 - Traffic-Aware Tabu: Cost=1065.01, Visited=50/50
Env 2 - Greedy-Traffic: Cost=1018.61, Visited=50/50
Env 2 - Random: Cost=1242.95, Visited=50/50
Env 2 - Largest-Demand: Cost=1162.54, Visited=50/50
Env 2 - Max-Reachable: Cost=1106.14, Visited=50/50
Env 2 - Clarke-Wright: Cost=1061.91, Visited=50/50
Env 2 - Sweep: Cost=1292.98, Visited=50/50


Env 3 - GNN: Cost=1121.91, Visited=50/50
Env 3 - Hybri

In [ ]:
import torch

def evaluate_hybrid_policy_clarke_wright(policy, env, k=3):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    state = env.reset().to(device)
    route = [env.depot]
    step_costs = []
    done = False
    max_steps = env.num_nodes * 10
    step = 0
    vehicle_capacity = env.vehicle_capacity

    # Use expected_demands if present, else fallback to demands
    demands = getattr(env, 'expected_demands', getattr(env, 'demands', None))
    if demands is None:
        raise ValueError("Environment missing 'expected_demands' or 'demands' attribute")
    actual_demands = getattr(env, 'actual_demands', demands)

    # Precompute savings at the start
    def compute_initial_savings():
        savings = []
        traffic_matrix = env.current_traffic_matrix
        for i in env.customer_nodes:
            for j in env.customer_nodes:
                if i < j and i not in env.visited and j not in env.visited:
                    if demands[i] + demands[j] <= vehicle_capacity:
                        s_ij = (traffic_matrix[env.depot][i] +
                                traffic_matrix[env.depot][j] -
                                traffic_matrix[i][j])
                        savings.append((s_ij, i, j))
        return sorted(savings, key=lambda x: x[0], reverse=True)

    savings = compute_initial_savings()
    routes = [[env.depot, i, env.depot] for i in env.customer_nodes]

    while not done and step < max_steps:
        with torch.no_grad():
            probs = policy(state).clone()

        current = env.current_node
        feasible_nodes = [
            node for node in env.customer_nodes
            if node not in env.visited and demands[node] <= env.vehicle_load
        ]

        if not feasible_nodes:
            action = env.depot
        else:
            # Update savings for current state
            savings = [
                (s, i, j) for s, i, j in savings
                if i not in env.visited and j not in env.visited
                and demands[i] + demands[j] <= env.vehicle_load
            ]

            # Find routes containing feasible nodes
            top_k_nodes = set()
            total_demand = 0
            i = 0
            while i < len(savings) and len(top_k_nodes) < k:
                s_ij, node_i, node_j = savings[i]
                route_i = next((r for r in routes if node_i in r[1:-1]), None)
                route_j = next((r for r in routes if node_j in r[1:-1]), None)
                if route_i and route_j and route_i != route_j:
                    if route_i[-2] == node_i and route_j[1] == node_j:
                        merged_demand = sum(demands[n] for n in route_i[1:-1] + route_j[1:-1])
                        if merged_demand <= env.vehicle_load:
                            if total_demand + demands[node_i] <= env.vehicle_load:
                                top_k_nodes.add(node_i)
                                total_demand += demands[node_i]
                            if total_demand + demands[node_j] <= env.vehicle_load:
                                top_k_nodes.add(node_j)
                                total_demand += demands[node_j]
                i += 1

            # Fallback to nearest feasible node if no savings-based nodes
            if not top_k_nodes:
                distances = [(node, env.current_traffic_matrix[current][node]) for node in feasible_nodes]
                top_k_nodes = {node for node, _ in sorted(distances, key=lambda x: x[1])[:k]}

            # Create mask for top-k nodes or depot
            greedy_mask = torch.tensor([
                1.0 if i in top_k_nodes or i == env.depot else 0.0
                for i in range(env.num_nodes)
            ], device=probs.device)

            masked_probs = probs * greedy_mask
            if masked_probs.sum() == 0:
                action = env.depot
            else:
                # Prioritize highest-savings node deterministically
                if top_k_nodes:
                    best_node = min(
                        top_k_nodes,
                        key=lambda n: env.current_traffic_matrix[current][n],
                        default=env.depot
                    )
                    if masked_probs[best_node] > 0:
                        action = best_node
                    else:
                        masked_probs /= masked_probs.sum()
                        action = torch.multinomial(masked_probs, 1).item()
                else:
                    masked_probs /= masked_probs.sum()
                    action = torch.multinomial(masked_probs, 1).item()

        # Check actual demand before taking action
        actual_demand = actual_demands.get(action, demands.get(action, 0))
        if actual_demand is None:
            actual_demand = 0  # Fallback to 0 if demand is missing
        if action != env.depot and actual_demand > env.vehicle_load:
            action = env.depot  # Return to depot to refill

        state, step_reward, done = env.step(action)
        route.append(action)
        step_costs.append(-step_reward)
        step += 1

        # Update routes if action was a customer
        if action != env.depot:
            for r in routes:
                if action in r[1:-1]:
                    routes.remove(r)
                    break
            savings = [s for s in savings if action not in (s[1], s[2])]

        # Check if depot return is needed
        feasible_remaining = any(
            node not in env.visited and demands[node] <= env.vehicle_load
            for node in env.customer_nodes
        )
        if not done and (not feasible_remaining or env.vehicle_load == 0):
            if env.current_node != env.depot:
                state, step_reward, done = env.step(env.depot)
                route.append(env.depot)
                step_costs.append(-step_reward)
                env.vehicle_load = vehicle_capacity
                step += 1

    # Ensure route ends with depot
    if route[-1] != env.depot:
        last = route[-1]
        step_costs.append(env.current_traffic_matrix[last][env.depot])
        route.append(env.depot)

    total_cost = sum(step_costs)
    return total_cost, len(env.visited), route, step_costs

In [ ]:
import time
import torch
import numpy as np
from copy import deepcopy
import pandas as pd
import matplotlib.pyplot as plt

def compare_methods(examples=10):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    policy = GNNPolicy().to(device)
    policy.load_state_dict(torch.load("gnn_policy_ppo_latest.pth", map_location=torch.device("mps"), weights_only=True))
    policy.eval()

    test_envs = [CVRPGridEnv(grid_size=10, vehicle_capacity=15, num_customers=50, seed=i) for i in range(examples)]

    # Store (cost, visited, time) tuples
    gnn_results = []
    hybrid_results = []
    hybrid_cw_results = []
    traffic_aware_results = []
    greedy_traffic_results = []
    random_results = []
    largest_demand_results = []
    max_reachable_results = []
    cw_results = []
    sweep_results = []

    print("Evaluating and visualizing on test environments...\n")
    for i, env in enumerate(test_envs):
        start = time.time()
        gnn_cost, gnn_visited, gnn_route, gnn_step_costs = evaluate_gnn_policy(policy, deepcopy(env))
        gnn_time = time.time() - start
        gnn_results.append((gnn_cost, gnn_visited, gnn_time))
        print(f"Env {i+1} - GNN: Cost={gnn_cost:.2f}, Visited={gnn_visited}/{env.num_customers}, Time={gnn_time:.3f}s")

        start = time.time()
        hybrid_cost, hybrid_visited, hybrid_route, hybrid_step_costs = evaluate_hybrid_policy(policy, deepcopy(env), k=3)
        hybrid_time = time.time() - start
        hybrid_results.append((hybrid_cost, hybrid_visited, hybrid_time))
        print(f"Env {i+1} - Hybrid GNN+Greedy: Cost={hybrid_cost:.2f}, Visited={hybrid_visited}/{env.num_customers}, Time={hybrid_time:.3f}s")

        start = time.time()
        hybrid_cw_cost, hybrid_cw_visited, hybrid_cw_route, hybrid_cw_step_costs = evaluate_hybrid_policy_clarke_wright(policy, deepcopy(env), k=3)
        hybrid_cw_time = time.time() - start
        hybrid_cw_results.append((hybrid_cw_cost, hybrid_cw_visited, hybrid_cw_time))
        print(f"Env {i+1} - Hybrid GNN+Clarke-Wright: Cost={hybrid_cw_cost:.2f}, Visited={hybrid_cw_visited}/{env.num_customers}, Time={hybrid_cw_time:.3f}s")

        start = time.time()
        traffic_cost, traffic_visited, traffic_route, traffic_step_costs = traffic_aware_tabu_search(deepcopy(env))
        traffic_time = time.time() - start
        traffic_aware_results.append((traffic_cost, traffic_visited, traffic_time))
        print(f"Env {i+1} - Traffic-Aware Tabu: Cost={traffic_cost:.2f}, Visited={traffic_visited}/{env.num_customers}, Time={traffic_time:.3f}s")

        start = time.time()
        greedy_ta_cost, greedy_ta_visited, greedy_ta_route, greedy_ta_step_costs = greedy_search_traffic_aware(deepcopy(env), return_route=True)
        greedy_ta_time = time.time() - start
        greedy_traffic_results.append((greedy_ta_cost, greedy_ta_visited, greedy_ta_time))
        print(f"Env {i+1} - Greedy-Traffic: Cost={greedy_ta_cost:.2f}, Visited={greedy_ta_visited}/{env.num_customers}, Time={greedy_ta_time:.3f}s")

        start = time.time()
        random_cost, random_visited, random_route, random_step_costs = random_search(deepcopy(env), return_route=True)
        random_time = time.time() - start
        random_results.append((random_cost, random_visited, random_time))
        print(f"Env {i+1} - Random: Cost={random_cost:.2f}, Visited={random_visited}/{env.num_customers}, Time={random_time:.3f}s")

        start = time.time()
        largest_cost, largest_visited, largest_route, largest_step_costs = largest_demand_search(deepcopy(env), return_route=True)
        largest_time = time.time() - start
        largest_demand_results.append((largest_cost, largest_visited, largest_time))
        print(f"Env {i+1} - Largest-Demand: Cost={largest_cost:.2f}, Visited={largest_visited}/{env.num_customers}, Time={largest_time:.3f}s")

        start = time.time()
        max_reach_cost, max_reach_visited, max_reach_route, max_reach_step_costs = max_reachable_search(deepcopy(env), return_route=True)
        max_reach_time = time.time() - start
        max_reachable_results.append((max_reach_cost, max_reach_visited, max_reach_time))
        print(f"Env {i+1} - Max-Reachable: Cost={max_reach_cost:.2f}, Visited={max_reach_visited}/{env.num_customers}, Time={max_reach_time:.3f}s")

        start = time.time()
        cw_cost, cw_visited, cw_route, cw_step_costs = clarke_wright(deepcopy(env), return_route=True)
        cw_time = time.time() - start
        cw_results.append((cw_cost, cw_visited, cw_time))
        print(f"Env {i+1} - Clarke-Wright: Cost={cw_cost:.2f}, Visited={cw_visited}/{env.num_customers}, Time={cw_time:.3f}s")

        start = time.time()
        sweep_cost, sweep_visited, sweep_route, sweep_step_costs = sweep(deepcopy(env), return_route=True)
        sweep_time = time.time() - start
        sweep_results.append((sweep_cost, sweep_visited, sweep_time))
        print(f"Env {i+1} - Sweep: Cost={sweep_cost:.2f}, Visited={sweep_visited}/{env.num_customers}, Time={sweep_time:.3f}s\n")

    def mean_se(values):
        values = np.array(values)
        mean = np.mean(values)
        se = np.std(values, ddof=1) / np.sqrt(len(values))
        return mean, se

    def summarize(name, results, env):
        costs = [r[0] for r in results]
        visited = [r[1] / env.num_customers * 100 for r in results]
        times = [r[2] for r in results]
        mean_cost, se_cost = mean_se(costs)
        mean_visited, se_visited = mean_se(visited)
        mean_time, se_time = mean_se(times)
        print(f"{name:<20} Avg Cost = {mean_cost:.2f} ± {se_cost:.2f}, "
              f"Avg Visited = {mean_visited:.2f}% ± {se_visited:.2f}%, "
              f"Avg Time = {mean_time:.3f}s ± {se_time:.3f}s")

    rows = []
    for i in range(examples):
        rows.append({
            "env": i + 1,
            "gnn_cost": gnn_results[i][0],
            "gnn_visited_pct": gnn_results[i][1] / test_envs[i].num_customers * 100,
            "gnn_time": gnn_results[i][2],
            "hybrid_cost": hybrid_results[i][0],
            "hybrid_visited_pct": hybrid_results[i][1] / test_envs[i].num_customers * 100,
            "hybrid_time": hybrid_results[i][2],
            "hybrid_cw_cost": hybrid_cw_results[i][0],
            "hybrid_cw_visited_pct": hybrid_cw_results[i][1] / test_envs[i].num_customers * 100,
            "hybrid_cw_time": hybrid_cw_results[i][2],
            "traffic_cost": traffic_aware_results[i][0],
            "traffic_visited_pct": traffic_aware_results[i][1] / test_envs[i].num_customers * 100,
            "traffic_time": traffic_aware_results[i][2],
            "greedy_traffic_cost": greedy_traffic_results[i][0],
            "greedy_traffic_visited_pct": greedy_traffic_results[i][1] / test_envs[i].num_customers * 100,
            "greedy_traffic_time": greedy_traffic_results[i][2],
            "random_cost": random_results[i][0],
            "random_visited_pct": random_results[i][1] / test_envs[i].num_customers * 100,
            "random_time": random_results[i][2],
            "largest_demand_cost": largest_demand_results[i][0],
            "largest_demand_visited_pct": largest_demand_results[i][1] / test_envs[i].num_customers * 100,
            "largest_demand_time": largest_demand_results[i][2],
            "max_reachable_cost": max_reachable_results[i][0],
            "max_reachable_visited_pct": max_reachable_results[i][1] / test_envs[i].num_customers * 100,
            "max_reachable_time": max_reachable_results[i][2],
            "cw_cost": cw_results[i][0],
            "cw_visited_pct": cw_results[i][1] / test_envs[i].num_customers * 100,
            "cw_time": cw_results[i][2],
            "sweep_cost": sweep_results[i][0],
            "sweep_visited_pct": sweep_results[i][1] / test_envs[i].num_customers * 100,
            "sweep_time": sweep_results[i][2],
        })

    df = pd.DataFrame(rows)
    df.to_csv("cvrp_comparison_results.csv", index=False)
    print("\nSaved detailed results to cvrp_comparison_results.csv")

    print("\n=== Performance Summary ===")
    summarize("GNN Policy", gnn_results, env)
    summarize("Hybrid GNN+Greedy", hybrid_results, env)
    summarize("Hybrid GNN+Clarke-Wright", hybrid_cw_results, env)
    summarize("Traffic-Aware Tabu", traffic_aware_results, env)
    summarize("Greedy-Traffic", greedy_traffic_results, env)
    summarize("Random", random_results, env)
    summarize("Largest-Demand", largest_demand_results, env)
    summarize("Max-Reachable", max_reachable_results, env)
    summarize("Clarke-Wright", cw_results, env)
    summarize("Sweep", sweep_results, env)

if __name__ == "__main__":
    compare_methods(examples=100)